In [ ]:
import pandas as pd

weather_data_cleaned = pd.read_csv('data/GlobalWeatherRepository_Cleaned.csv')
weather_data_scaled = pd.read_csv('data/GlobalWeatherRepository_Cleaned_Scaled.csv')

**🛠️ Feature Engineering Plan**

| # | Feature | Description |
|---|---|---|
| 1 | `season` | Extract season from month (Winter/Spring/Summer/Fall) |
| 2 | `temp_range` | Difference between temperature and feels_like |
| 3 | `heat_index` | Combined temp + humidity indicator |
| 4 | `wind_chill` | How cold it actually feels with wind |
| 5 | `is_rainy` | Binary flag — precipitation > 0 |
| 6 | `air_quality_index` | Combined AQ score from all pollutants |
| 7 | `time_of_day` | Extract from `last_updated` (Morning/Afternoon/Evening/Night) |
| 8 | `is_daytime` | Binary flag from sunrise/sunset |

In [18]:
# Parse datetime
weather_data_cleaned['last_updated'] = pd.to_datetime(weather_data_cleaned['last_updated'])

# Season from month
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

weather_data_cleaned['season'] = weather_data_cleaned['last_updated'].dt.month.apply(get_season)

# Temp range (comfort gap)
weather_data_cleaned['temp_range'] = (
    weather_data_cleaned['temperature_celsius'] - weather_data_cleaned['feels_like_celsius']
)

print("Season value counts:")
print(weather_data_cleaned['season'].value_counts())
print(f"\nTemp range stats:")
print(weather_data_cleaned['temp_range'].describe().round(2))

Season value counts:
season
Fall      35435
Summer    35365
Winter    34849
Spring    32346
Name: count, dtype: int64

Temp range stats:
count    137995.00
mean         -0.85
std           2.69
min         -17.90
25%          -2.40
50%           0.00
75%           0.60
max          12.50
Name: temp_range, dtype: float64


Insights:

- Season distribution — fairly balanced across all four seasons (~32K–35K records each), meaning the dataset has good temporal coverage with no season being significantly over or underrepresented. ✅
- Temp range mean of -0.85°C — on average, it feels slightly colder than the actual temperature, which makes sense as wind and humidity typically reduce perceived warmth.
- Min of -17.9°C — extreme wind chill cases (likely Arctic/high-altitude locations) where feels-like is dramatically lower than actual temperature.
- Max of +12.5°C — high humidity locations where it feels much hotter than actual temperature (heat island effect in tropical cities).
- Median of 0.0°C — exactly half the readings have no difference between actual and feels-like temperature. ✅



In [19]:
# Verify which dataframe is being used
print(weather_data_cleaned['wind_mph'].describe().round(2))
print(weather_data_cleaned['humidity'].describe().round(2))

count    137995.00
mean          7.95
std           5.04
min           2.20
25%           3.80
50%           6.70
75%          11.00
max          23.30
Name: wind_mph, dtype: float64
count    137995.00
mean         66.69
std          23.86
min           2.00
25%          51.00
50%          72.00
75%          86.00
max         100.00
Name: humidity, dtype: float64


In [20]:
weather_data_cleaned['heat_index'] = np.where(
    (weather_data_cleaned['temperature_celsius'] > 27) & 
    (weather_data_cleaned['humidity'] > 40),
    weather_data_cleaned['temperature_celsius'] +
    (0.33 * weather_data_cleaned['humidity']) -
    (0.70 * weather_data_cleaned['wind_mph']) - 4.00,
    weather_data_cleaned['temperature_celsius']
)

weather_data_cleaned['wind_chill'] = np.where(
    (weather_data_cleaned['temperature_celsius'] < 10) & 
    (weather_data_cleaned['wind_mph'] > 3),
    13.12 +
    0.6215 * weather_data_cleaned['temperature_celsius'] -
    11.37 * (weather_data_cleaned['wind_mph'] ** 0.16) +
    0.3965 * weather_data_cleaned['temperature_celsius'] * (weather_data_cleaned['wind_mph'] ** 0.16),
    weather_data_cleaned['temperature_celsius']
)

print("Heat Index stats (fixed):")
print(weather_data_cleaned['heat_index'].describe().round(2))
print("\nWind Chill stats (fixed):")
print(weather_data_cleaned['wind_chill'].describe().round(2))

Heat Index stats (fixed):
count    137995.00
mean         23.93
std          12.98
min         -29.80
25%          15.80
50%          23.90
75%          33.00
max          56.01
Name: heat_index, dtype: float64

Wind Chill stats (fixed):
count    137995.00
mean         21.04
std          10.15
min         -34.34
25%          15.80
50%          23.90
75%          28.00
max          49.20
Name: wind_chill, dtype: float64


Insights:

- Heat Index min of -29.80°C — this is now just the fallback temperature_celsius value for cold locations (our np.where condition), not a formula error. ✅
- Heat Index max of 56.01°C — realistic for extremely hot and humid locations like the Gulf region. ✅
- Heat Index mean of 23.93°C — close to the actual temperature mean, confirming the conditional logic is working correctly. ✅
- Wind Chill range [-34.34, 49.20°C] — sensible range, with the max again being the fallback actual temperature for warm locations. ✅



In [21]:
# #5 - Binary rainy flag
weather_data_cleaned['is_rainy'] = (weather_data_cleaned['precip_mm'] > 0).astype(int)

# #6 - Combined air quality index (normalized average of all pollutants)
aq_cols = [
    'air_quality_Carbon_Monoxide', 'air_quality_Ozone',
    'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide',
    'air_quality_PM2.5', 'air_quality_PM10'
]

# Normalize each to 0-1 then average
from sklearn.preprocessing import MinMaxScaler

aq_scaler = MinMaxScaler()
aq_normalized = aq_scaler.fit_transform(weather_data_cleaned[aq_cols])
weather_data_cleaned['air_quality_index'] = aq_normalized.mean(axis=1)

print("Is Rainy value counts:")
print(weather_data_cleaned['is_rainy'].value_counts())
print(f"\nRainy percentage: {weather_data_cleaned['is_rainy'].mean()*100:.1f}%")
print("\nAir Quality Index stats:")
print(weather_data_cleaned['air_quality_index'].describe().round(4))

Is Rainy value counts:
is_rainy
0    92121
1    45874
Name: count, dtype: int64

Rainy percentage: 33.2%

Air Quality Index stats:
count    137995.0000
mean          0.5409
std           0.0659
min           0.2782
25%           0.5000
50%           0.5259
75%           0.5625
max           0.9772
Name: air_quality_index, dtype: float64


Insights:

- 33.2% of readings are rainy — meaning 1 in 3 weather snapshots recorded some precipitation. This is a realistic global average. ✅
- 66.8% dry readings — consistent with our earlier finding that most precip_mm values were 0. ✅
- Air Quality Index mean of 0.54 — sitting just above the midpoint of the 0–1 scale, suggesting moderate pollution levels globally on average.
- Tight std of 0.066 — the combined index is quite stable, meaning no single pollutant is dramatically dominating the score. ✅
- Min of 0.28 — cleanest air readings, likely remote/rural locations.
- Max of 0.977 — near-maximum pollution, likely industrial cities in China/India during peak pollution events. ✅

In [22]:
# #7 - Time of day from last_updated
def get_time_of_day(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

weather_data_cleaned['time_of_day'] = (
    weather_data_cleaned['last_updated'].dt.hour.apply(get_time_of_day)
)

print("Time of Day distribution:")
print(weather_data_cleaned['time_of_day'].value_counts())

Time of Day distribution:
time_of_day
Morning      66454
Afternoon    39594
Night        18889
Evening      13058
Name: count, dtype: int64


- Time of Day distribution — Morning dominates (48%), which suggests the dataset is collected primarily during morning observation windows. Worth noting as a data collection bias. ✅


In [24]:
# Check raw sunrise/sunset format
print("Sunrise sample values:")
print(weather_data_cleaned['sunrise'].value_counts().head(10))
print("\nSunset sample values:")
print(weather_data_cleaned['sunset'].value_counts().head(10))

Sunrise sample values:
sunrise
2026-05-27 05:57:00    1459
2026-05-27 06:10:00    1459
2026-05-27 06:11:00    1438
2026-05-27 05:56:00    1428
2026-05-27 05:55:00    1422
2026-05-27 06:17:00    1422
2026-05-27 05:58:00    1407
2026-05-27 06:21:00    1399
2026-05-27 06:24:00    1395
2026-05-27 05:54:00    1389
Name: count, dtype: int64

Sunset sample values:
sunset
2026-05-27 18:02:00    1369
2026-05-27 18:17:00    1292
2026-05-27 18:03:00    1286
2026-05-27 18:11:00    1273
2026-05-27 18:18:00    1270
2026-05-27 18:28:00    1260
2026-05-27 18:19:00    1254
2026-05-27 18:13:00    1214
2026-05-27 18:35:00    1212
2026-05-27 18:29:00    1201
Name: count, dtype: int64


In [25]:
# Parse sunrise and sunset as full datetime
weather_data_cleaned['sunrise_hour'] = pd.to_datetime(
    weather_data_cleaned['sunrise'], errors='coerce'
).dt.hour

weather_data_cleaned['sunset_hour'] = pd.to_datetime(
    weather_data_cleaned['sunset'], errors='coerce'
).dt.hour

# Apply is_daytime
current_hour = weather_data_cleaned['last_updated'].dt.hour

weather_data_cleaned['is_daytime'] = (
    (current_hour >= weather_data_cleaned['sunrise_hour']) &
    (current_hour < weather_data_cleaned['sunset_hour'])
).astype(int)

print("Sunrise hour sample:")
print(weather_data_cleaned['sunrise_hour'].value_counts().head(5))
print("\nSunset hour sample:")
print(weather_data_cleaned['sunset_hour'].value_counts().head(5))
print(f"\nDaytime readings: {weather_data_cleaned['is_daytime'].sum():,} ({weather_data_cleaned['is_daytime'].mean()*100:.1f}%)")
print(f"Nighttime readings: {(weather_data_cleaned['is_daytime']==0).sum():,} ({(1-weather_data_cleaned['is_daytime'].mean())*100:.1f}%)")

Sunrise hour sample:
sunrise_hour
6    64576
5    41849
7    21365
8     5108
4     3987
Name: count, dtype: int64

Sunset hour sample:
sunset_hour
18    60811
17    32809
19    19909
20    10403
16     7750
Name: count, dtype: int64

Daytime readings: 106,587 (77.2%)
Nighttime readings: 31,408 (22.8%)


Insights:

- Sunrise hours 5–7am — perfectly realistic globally, with 6am being the most common sunrise hour. ✅
- Sunset hours 17–19 — consistent with real sunset times across different latitudes and seasons. ✅
- 77.2% daytime readings — makes sense given the time_of_day distribution we saw earlier, where Morning and Afternoon dominated the dataset. ✅
- 22.8% nighttime readings — captured during Evening and Night observation windows. ✅




---

**🛠️ Remaining Feature Engineering Actions:**

| # | Action | Status |
|---|---|---|
| 1 | Condition text normalization + class collapsing | ❌ Todo |
| 2 | Drop `feels_like_celsius` & `gust_mph` | ❌ Todo |
| 3 | PCA on air quality columns | ❌ Todo |
| 4 | UV Index binning | ❌ Todo |
| 5 | Yeo-Johnson transform on skewed features | ❌ Todo |  
  
  
---

In [26]:
# Normalize casing
weather_data_cleaned['condition_text'] = weather_data_cleaned['condition_text'].str.strip().str.title()

# Check before collapsing
print("Top 20 conditions before collapsing:")
print(weather_data_cleaned['condition_text'].value_counts().head(20))

Top 20 conditions before collapsing:
condition_text
Partly Cloudy                          48469
Sunny                                  39670
Patchy Rain Nearby                     11498
Overcast                                7489
Clear                                   7292
Mist                                    5791
Light Rain                              4312
Light Rain Shower                       3153
Fog                                     1728
Cloudy                                  1650
Moderate Or Heavy Rain With Thunder     1104
Moderate Rain                           1047
Patchy Light Rain With Thunder           855
Light Drizzle                            683
Light Snow                               523
Patchy Light Drizzle                     459
Moderate Or Heavy Rain Shower            288
Thundery Outbreaks In Nearby             270
Freezing Fog                             248
Patchy Light Rain                        244
Name: count, dtype: int64


In [29]:
# Define precipitation event conditions
precipitation_conditions = [
    'Patchy Rain Nearby', 'Light Rain', 'Light Rain Shower',
    'Moderate Or Heavy Rain With Thunder', 'Moderate Rain',
    'Patchy Light Rain With Thunder', 'Light Drizzle',
    'Patchy Light Drizzle', 'Moderate Or Heavy Rain Shower',
    'Thundery Outbreaks In Nearby', 'Patchy Light Rain',
    'Heavy Rain', 'Heavy Rain At Times', 'Light Sleet',
    'Moderate Or Heavy Sleet', 'Ice Pellets', 'Light Sleet Showers',
    'Moderate Or Heavy Snow With Thunder', 'Blizzard',
    'Light Snow', 'Moderate Snow', 'Heavy Snow',
    'Patchy Snow Nearby', 'Patchy Light Snow',
    'Blowing Snow', 'Freezing Drizzle', 'Heavy Freezing Drizzle',
    'Light Freezing Rain', 'Moderate Or Heavy Freezing Rain',
    'Torrential Rain Shower'    'Patchy Light Rain In Area With Thunder',
    'Moderate Rain At Times','Light Snow Showers','Moderate Or Heavy Snow Showers',
    'Patchy Rain Possible','Patchy Heavy Snow','Thundery Outbreaks Possible',
    'Patchy Moderate Snow','Moderate Or Heavy Rain In Area With Thunder',
    'Patchy Snow Possible','Patchy Light Snow In Area With Thunder',
    'Moderate Or Heavy Snow In Area With Thunder','Patchy Light Rain In Area With Thunder', 
    'Torrential Rain Shower'
]

weather_data_cleaned['condition_grouped'] = weather_data_cleaned['condition_text'].apply(
    lambda x: 'Precipitation_Event' if x in precipitation_conditions else x
)

print("Condition distribution after collapsing:")
print(weather_data_cleaned['condition_grouped'].value_counts())
print(f"\nTotal unique conditions before: {weather_data_cleaned['condition_text'].nunique()}")
print(f"Total unique conditions after:  {weather_data_cleaned['condition_grouped'].nunique()}")

Condition distribution after collapsing:
condition_grouped
Partly Cloudy          48469
Sunny                  39670
Precipitation_Event    25658
Overcast                7489
Clear                   7292
Mist                    5791
Fog                     1728
Cloudy                  1650
Freezing Fog             248
Name: count, dtype: int64

Total unique conditions before: 48
Total unique conditions after:  9


Insights:
- Perfect! 9 clean, meaningful condition classes ✅
- Precipitation_Event is now the 3rd largest class with 25,658 records — enough for models to learn from effectively ✅
- Class distribution looks reasonable — no extreme imbalance between the top classes


In [30]:
# Drop near-perfectly correlated columns
cols_to_drop = [
    'feels_like_celsius',       # r=0.98 with temperature_celsius
    'feels_like_fahrenheit',    # unit duplicate of above
    'gust_mph',                 # r=0.95 with wind_mph
    'gust_kph',                 # unit duplicate of above
    'temperature_fahrenheit',   # unit duplicate of temperature_celsius
    'wind_kph',                 # unit duplicate of wind_mph
    'pressure_in',              # unit duplicate of pressure_mb
    'precip_in',                # unit duplicate of precip_mm
    'visibility_miles',         # unit duplicate of visibility_km
]

weather_data_cleaned.drop(columns=cols_to_drop, inplace=True)

print(f"Columns dropped: {len(cols_to_drop)}")
print(f"Remaining columns: {weather_data_cleaned.shape[1]}")
print("\nRemaining column list:")
print(weather_data_cleaned.columns.tolist())

Columns dropped: 9
Remaining columns: 43

Remaining column list:
['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'condition_text', 'wind_mph', 'wind_degree', 'wind_direction', 'pressure_mb', 'precip_mm', 'humidity', 'cloud', 'visibility_km', 'uv_index', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination', 'season', 'temp_range', 'heat_index', 'wind_chill', 'is_rainy', 'air_quality_index', 'time_of_day', 'sunrise_hour', 'sunset_hour', 'is_daytime', 'condition_grouped']


In [31]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Air quality columns to compress
aq_cols = [
    'air_quality_Carbon_Monoxide', 'air_quality_Ozone',
    'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide',
    'air_quality_PM2.5', 'air_quality_PM10'
]

# Standardize before PCA
aq_scaler = StandardScaler()
aq_scaled = aq_scaler.fit_transform(weather_data_cleaned[aq_cols])

# Fit PCA
pca = PCA()
pca.fit(aq_scaled)

# Check explained variance
explained_variance = pca.explained_variance_ratio_.cumsum()
print("Cumulative Explained Variance by Component:")
for i, var in enumerate(explained_variance, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.1f}%)")

Cumulative Explained Variance by Component:
  PC1: 0.4794 (47.9%)
  PC2: 0.6896 (69.0%)
  PC3: 0.8253 (82.5%)
  PC4: 0.9311 (93.1%)
  PC5: 0.9822 (98.2%)
  PC6: 1.0000 (100.0%)


Insights:

- PC1 alone captures 47.9% — lower than expected for highly correlated features, suggesting the pollutants are moderately rather than perfectly correlated.
- PC1 + PC2 = 69% — two components still leave 31% unexplained.
- PC1 + PC2 + PC3 = 82.5% — three components cross the 80% threshold, which is the standard cutoff. ✅

In [32]:
# Apply PCA with 3 components
pca_final = PCA(n_components=3)
aq_pca = pca_final.fit_transform(aq_scaled)

# Add to dataframe
weather_data_cleaned['pollution_PC1'] = aq_pca[:, 0]
weather_data_cleaned['pollution_PC2'] = aq_pca[:, 1]
weather_data_cleaned['pollution_PC3'] = aq_pca[:, 2]

# Drop original air quality columns
weather_data_cleaned.drop(columns=aq_cols, inplace=True)

# Also drop index columns since we have PCA now
weather_data_cleaned.drop(columns=[
    'air_quality_us-epa-index', 
    'air_quality_gb-defra-index',
    'air_quality_index'  # our manually created one earlier
], inplace=True, errors='ignore')

print(f"Remaining columns: {weather_data_cleaned.shape[1]}")
print("\nNew pollution PCA features:")
print(weather_data_cleaned[['pollution_PC1', 'pollution_PC2', 'pollution_PC3']].describe().round(3))

Remaining columns: 37

New pollution PCA features:
       pollution_PC1  pollution_PC2  pollution_PC3
count     137995.000     137995.000     137995.000
mean           0.000          0.000         -0.000
std            1.696          1.123          0.902
min         -102.351        -11.151       -222.097
25%           -1.001         -0.643         -0.357
50%           -0.567         -0.118         -0.026
75%            0.269          0.497          0.342
max           11.264         54.180          9.591


Insights:

- Means of ~0 — confirms PCA standardization worked correctly. ✅
- PC1 std of 1.696 — highest variance component, capturing the dominant pollution signal across all 6 pollutants. ✅
- PC1 min of -102 and PC3 min of -222 🚨 — extreme negative values suggest a few records are very far from the pollution centroid. These are likely the same industrial hotspot cities (China, India) driving outliers in the original features. They won't cause issues for tree-based models but worth noting.
- PC2 max of 54.18 — a few extreme positive outliers in the second pollution dimension, likely capturing ozone-specific spikes. ✅



In [33]:
# Bin UV index into WHO standard ordinal tiers
def bin_uv(uv):
    if uv <= 2: return 'Low'
    elif uv <= 5: return 'Moderate'
    elif uv <= 7: return 'High'
    elif uv <= 10: return 'Very High'
    else: return 'Extreme'

weather_data_cleaned['uv_category'] = weather_data_cleaned['uv_index'].apply(bin_uv)

# Drop original uv_index
weather_data_cleaned.drop(columns=['uv_index'], inplace=True)

print("UV Category distribution:")
print(weather_data_cleaned['uv_category'].value_counts())
print(f"\nRemaining columns: {weather_data_cleaned.shape[1]}")

UV Category distribution:
uv_category
Low          71049
Moderate     25476
High         19523
Very High    14979
Extreme       6968
Name: count, dtype: int64

Remaining columns: 37


Insights:

- Low UV dominates at 71,049 (51.5%) — consistent with our earlier finding that most readings are captured during morning hours or overcast conditions. ✅
- Extreme UV at 6,968 (5%) — realistic representation of high-altitude and equatorial regions. ✅
- Good ordinal spread — all 5 categories have sufficient records for modeling. ✅



In [34]:
from sklearn.preprocessing import PowerTransformer

# Features with heavy skew identified during EDA
skewed_cols = [
    'precip_mm',
    'wind_mph',
    'pressure_mb',
    'visibility_km'
]

# Check skewness before
print("Skewness BEFORE Yeo-Johnson:")
for col in skewed_cols:
    print(f"  {col}: {weather_data_cleaned[col].skew():.4f}")

# Apply Yeo-Johnson
pt = PowerTransformer(method='yeo-johnson')
weather_data_cleaned[skewed_cols] = pt.fit_transform(weather_data_cleaned[skewed_cols])

# Check skewness after
print("\nSkewness AFTER Yeo-Johnson:")
for col in skewed_cols:
    print(f"  {col}: {weather_data_cleaned[col].skew():.4f}")

Skewness BEFORE Yeo-Johnson:
  precip_mm: 4.0302
  wind_mph: 0.9519
  pressure_mb: 0.0391
  visibility_km: 1.0467

Skewness AFTER Yeo-Johnson:
  precip_mm: 1.5061
  wind_mph: 0.0075
  pressure_mb: 0.0002
  visibility_km: 0.8094


Insights:

- precip_mm — skewness reduced from 4.03 → 1.51, a significant improvement but still moderately skewed. This is expected given the zero-inflation (most days have no rain). Acceptable for modeling. ✅
- wind_mph — nearly perfectly normalized, 0.95 → 0.007. Excellent transformation. ✅
- pressure_mb — was already near-normal (0.039) and is now essentially perfect (0.0002). ✅
- visibility_km — improved from 1.05 → 0.81, moderate improvement. The 10km cap we noted earlier limits how much we can normalize this column. Acceptable. ✅




---

**✅ Complete Feature Engineering Summary:**

| # | Action | Details | Status |
|---|---|---|---|
| 1 | Season extraction | Winter/Spring/Summer/Fall from month | ✅ |
| 2 | Temp range | Actual minus feels-like temperature | ✅ |
| 3 | Heat index | Conditional formula (temp>27°C & humidity>40%) | ✅ |
| 4 | Wind chill | Conditional formula (temp<10°C & wind>3mph) | ✅ |
| 5 | Is rainy flag | Binary — precip_mm > 0 | ✅ |
| 6 | Air quality index | Normalized average of 6 pollutants | ✅ |
| 7 | Time of day | Morning/Afternoon/Evening/Night | ✅ |
| 8 | Is daytime flag | Binary — between sunrise and sunset | ✅ |
| 9 | Condition grouping | 48 classes → 9 meaningful categories | ✅ |
| 10 | Drop redundant columns | 9 unit-duplicate columns removed | ✅ |
| 11 | PCA on air quality | 6 pollutants → 3 principal components | ✅ |
| 12 | UV index binning | Continuous → 5 WHO ordinal tiers | ✅ |
| 13 | Yeo-Johnson transform | Skewness reduced on 4 skewed features | ✅ |

---

In [ ]:
# Save dataframe with engineered features
weather_data_cleaned.to_csv('data/GlobalWeatherRepository_Engineered.csv', index=False)